# Week 1 · Day 1: Your First LLM Call

About **15 minutes**. You will:
1. set up your Groq key,
2. make your first call to an LLM,
3. learn the few settings that matter,
4. see why an LLM alone needs **tools** (that's Day 2).

Run the cells from top to bottom with **Shift + Enter**.

## Step 1 · Setup
1. Install the packages (the next cell does it; you only need to do this once).
2. Get a free key at **console.groq.com** → API Keys.
3. Give the notebook your key:
   - **On Google Colab:** click the 🔑 **Secrets** icon in the left bar, add a secret named `GROQ_API_KEY`, paste your key as the value, and switch **Notebook access** on.
   - **On your own computer:** make a file called **`.env`** in this folder (or any folder above it) containing the line `GROQ_API_KEY=your_key_here`.

Your key is like a password: **never share it** or put it in the notebook.

In [ ]:
%pip install -q groq python-dotenv requests

In [ ]:
import os
from dotenv import find_dotenv, load_dotenv
from groq import Groq
from IPython.display import Markdown, display   # shows answers as formatted text

load_dotenv(find_dotenv(usecwd=True), override=True)   # on your computer: reads your .env file
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")

try:                                                    # on Google Colab: reads the key from the 🔑 Secrets panel
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY") or GROQ_API_KEY
except Exception:                                       # not on Colab, or the secret isn't set / shared yet
    pass

MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")   # the model on the Day 1 slide; names change, check console.groq.com

print("Groq key:", "OK" if GROQ_API_KEY else "MISSING: add GROQ_API_KEY (Colab: the 🔑 Secrets panel · computer: a .env file), then run this cell again")
print("Model   :", MODEL)
client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

## Step 2 · Your first LLM call
One question in, one answer out. This is a plain **LLM call**: no tools, no loop, so it's **not an agent yet**.

Two small helpers we'll reuse below:
- `ask()` sends a question and returns the answer. It always adds a short **system message** asking for a brief answer, because gpt-oss models like to write a whole page.
- `show()` displays the answer as **formatted text** (Markdown), like a chat app does: tables become tables and `**bold**` becomes **bold**. `print()` would show the raw symbols instead.

In [ ]:
SHORT = {"role": "system", "content": "Answer briefly, in a few short sentences."}


def ask(question, **settings):
    """Send a question (or a list of messages) to the model and return its answer."""
    if client is None:
        return "(no Groq key yet: see Step 1)"
    messages = [{"role": "user", "content": question}] if isinstance(question, str) else question
    reply = client.chat.completions.create(model=MODEL, messages=[SHORT] + messages, **settings)
    return reply.choices[0].message.content


def show(answer):
    """Display an answer as formatted text (Markdown), like a chat app."""
    display(Markdown(answer or "*(empty answer)*"))


show(ask("Say hello to a class of AI students in one short sentence."))

## Step 3 · The settings that matter
The Groq Playground shows many settings. For now, only these matter:

| Setting | What it means |
|---|---|
| `model` | Which AI answers, e.g. `openai/gpt-oss-120b` |
| `messages` | The **whole conversation**, sent every time. Each message has a **role**: `system` (your rules), `user` (the person), `assistant` (the model's earlier replies) |
| `temperature` | How random the answer is: **0** = focused and repeatable, **2** = very random. Default 1 |
| `max_completion_tokens` | The longest answer allowed, in tokens (a token is about ¾ of a word) |
| `reasoning_effort` | For gpt-oss models: how long it thinks before answering, `low` / `medium` / `high` |

Everything else (Top P, Stream, Stop, Seed, JSON mode) can stay at the default for now. **Tools** come in Day 2.

## Step 4 · Try it

### `temperature`
The same question three times. Low temperature gives similar answers; high gives more variety.

In [ ]:
question = "Invent a name for a pet robot. Reply with the name only."
print("temperature 0  :", [ask(question, temperature=0) for _ in range(3)])
print("temperature 1.5:", [ask(question, temperature=1.5) for _ in range(3)])

### `messages`: the model has no memory
Every call starts from zero. The model only knows what you **send** in `messages`. Keeping track of the conversation is called **state** (Day 3).

In [ ]:
show(ask("My name is Asha. Please remember it."))
show(ask("What is my name?"))         # a NEW conversation: it doesn't know

history = [
    {"role": "user", "content": "My name is Asha. Please remember it."},
    {"role": "assistant", "content": "Nice to meet you, Asha!"},
    {"role": "user", "content": "What is my name?"},
]
show(ask(history))                    # with the history: now it knows

## Step 5 · The problem: an LLM alone needs tools
An LLM has **no internet** and **no calculator**. Read these two answers carefully.

In [ ]:
show(ask("What's the weather in Pokhara right now?"))
show(ask("What is 4,817 × 2,936? Reply with just the number."))
print("Exact answer:", f"{4817 * 2936:,}")

Live data and exact maths need **tools**. In Day 2 we build our own: weather, time, and a calculator.

## Bonus (optional) · A built-in tool
The gpt-oss models on Groq have a built-in **browser search**. Here Groq runs the search for us.

It works, but **we can't see or control** what happened inside. In Day 2 we build our own tools, so we can see every step.

**Always check the answer**, even when a tool was used: look at the date and the source. Search results can be out of date.

## Ready for Day 2?
- [ ] Step 1 says **Groq key: OK**
- [ ] Step 2 showed a hello message
- [ ] You can explain **temperature** and **messages** in your own words
- [ ] **Bring one question** an LLM alone can't answer, e.g. *"What is the USD to NPR rate today?"*

| Problem | Fix |
|---|---|
| Answers show `**`, `###` or `|` symbols | Display them with `show(...)` instead of `print(...)` |
| Very long answers | Re-run the `ask()` cell in Step 2: its system message asks for brief answers |
| `No module named 'groq'` | Run the install cell, then restart the kernel |
| `Groq key: MISSING` on Colab | Add a secret named `GROQ_API_KEY` in the 🔑 Secrets panel and switch **Notebook access** on, then run the key cell again |
| `Groq key: MISSING` on your computer | The file must be named exactly `.env` and sit in this folder or a folder above it; restart the kernel |
| `model not found` | Pick a current model in the Groq console and set `GROQ_MODEL` in `.env` |